In [1]:
from QbitSim import *
from torch import nn
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.maskable.utils import get_action_masks
from stable_baselines3.common.vec_env import DummyVecEnv
from IPython.display import clear_output

In [2]:
max_steps = 12
N_qbits = 3
allowed_gates = "H X Z M CX"

env = QbitEnv(N_qbits = N_qbits, allowed_gates = allowed_gates, max_steps=max_steps, action_repetition=None, max_cnots=2)

policy_kwargs = dict(net_arch=dict(pi=[128, 128], vf=[128, 128]))
model = MaskablePPO("MlpPolicy", env, policy_kwargs=policy_kwargs, verbose=1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [3]:
best_rew_mean = -100.
n_eval_episodes = 100
frames_per_session = 5_000

try:
    while True:
        clear_output(wait=True)
        model.learn(frames_per_session)
        rew_mean, var = evaluate_policy(model, env, n_eval_episodes=n_eval_episodes, reward_threshold=-100., warn=False)
        if rew_mean > best_rew_mean:
            best_rew_mean = rew_mean
            model.save("models/ppo_mask_2")
except KeyboardInterrupt:
    clear_output(wait=True)
    model.learn(frames_per_session)
    rew_mean, var = evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-100., warn=False)
    if rew_mean > best_rew_mean:
            best_rew_mean = rew_mean
            model.save("models/ppo_mask_2")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 12       |
|    ep_rew_mean     | -16      |
| time/              |          |
|    fps             | 2181     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 12          |
|    ep_rew_mean          | -16         |
| time/                   |             |
|    fps                  | 1814        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.035311177 |
|    clip_fraction        | 0.256       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.897      |
|    explained_variance   | 1           |
|    learning_rate        | 0.

In [6]:
test_model = MaskablePPO.load("models/ppo_mask")
rew_mean, var = evaluate_policy(test_model, env, n_eval_episodes=20, reward_threshold=-20, warn=False)
print(f"Mean reward = {rew_mean:.2f}")
obs, _ = env.reset()
for _ in range(max_steps):
    # Retrieve current action mask
    action_masks = get_action_masks(env)
    action, _states = test_model.predict(obs, action_masks=action_masks)
    obs, reward, terminated, truncated, info = env.step(action)
    print(info)
    if terminated:
        print("Success!")
        break
if not terminated:
    print("Failure!")

Mean reward = -7.00
{'Step': 0, 'M': None, 'Action': 'CX->[2, 1]', 'Terminal': False}
{'Step': 1, 'M': None, 'Action': 'H->[0]', 'Terminal': False}
{'Step': 2, 'M': None, 'Action': 'H->[2]', 'Terminal': False}
{'Step': 3, 'M': None, 'Action': 'CX->[0, 1]', 'Terminal': False}
{'Step': 4, 'M': 1, 'Action': 'M->[2]', 'Terminal': False}
{'Step': 5, 'M': 1, 'Action': 'M->[1]', 'Terminal': False}
{'Step': 6, 'M': None, 'Action': 'X->[0]', 'Terminal': False}
{'Step': 7, 'M': None, 'Action': 'Z->[0]', 'Terminal': True}
Success!


In [ ]:
model.save("models/ppo_mask")
del model # remove to demonstrate saving and loading

model = MaskablePPO.load("models/ppo_mask")